# wandb-finish — ex2: guarantee wandb.finish even on training-loop exception

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-finish`. Running the final beacon cell reports progress against the `Logging: wandb.finish` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.finish` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-finish`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-finish"
DD_SUBTOPIC = "Logging: wandb.finish"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `wandb.finish()` with try/finally — quick refresher

If your training loop raises (CUDA OOM, dataset NaN, you Ctrl-C), the default ARENA `train()` skips `wandb.finish` — the run sits in `running` state on the dashboard forever. Worse, the next `wandb.init` may silently append to the dead run instead of opening a new one.

**The fix is one keyword pair:**

```python
wandb.init(...)
try:
    for step in range(n_steps):
        train_step(...)
finally:
    wandb.finish()
```

**`finally` runs even on exception.** Whether the loop completes normally or blows up, `wandb.finish` is called exactly once. The exception still propagates after the finally block — you don't swallow it.

### Exercise 2 — guarantee wandb.finish even on training-loop exception

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the failure mode of an init/finish lifecycle that lacks try/finally and repair it so `wandb.finish` always runs, even when the inner loop raises.
> Keywords: wandb, finish, try-finally, exception-safety, mock
> ```

**KCs targeted:** `wandb-finish-after-train`, `try-finally-cleanup`

Implement `ex2_safe_train(n_steps, raise_at)`. A wandb-instrumented fake-train that uses try/finally to guarantee `finish` runs:

1. Call `wandb.init(project='arena', name='safe-run')`.
2. Wrap a `for step in range(n_steps)` loop in `try:` ... `finally: wandb.finish()`.
3. Inside the loop, if `raise_at is not None and step == raise_at`, raise `RuntimeError(f'simulated failure at step {step}')`.
4. Otherwise the loop is a no-op.
5. **Return** the number of steps completed BEFORE the raise (or `n_steps` if no raise happened). The exception must still propagate out of the function — do NOT swallow it; the test expects it via `with pytest.raises(...)`-style assert.

The test will run TWO scenarios:
- `raise_at=None` — normal completion. Assert `wandb.init` + `wandb.finish` each called once.
- `raise_at=2` — exception path. Assert RuntimeError propagated, `wandb.finish` STILL called exactly once.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex2_safe_train(n_steps: int, raise_at) -> int:
    """Init wandb, fake-train, finish in try/finally. Returns steps completed."""
    raise NotImplementedError()


def _test_ex2():
    # Scenario 1 — normal completion.
    wandb.init.reset_mock(); wandb.finish.reset_mock()
    out = ex2_safe_train(n_steps=5, raise_at=None)
    assert out == 5, f'normal path must return n_steps, got {out}'
    assert wandb.init.call_count == 1, f'init expected 1x, got {wandb.init.call_count}'
    assert wandb.finish.call_count == 1, f'finish expected 1x, got {wandb.finish.call_count}'

    # Scenario 2 — raise mid-loop. Exception must propagate, finish must STILL run.
    wandb.init.reset_mock(); wandb.finish.reset_mock()
    raised = False
    try:
        ex2_safe_train(n_steps=5, raise_at=2)
    except RuntimeError as e:
        raised = True
        assert 'simulated failure at step 2' in str(e), f'wrong message: {e}'
    assert raised, 'RuntimeError must propagate out of ex2_safe_train — do not swallow it'
    assert wandb.init.call_count == 1
    assert wandb.finish.call_count == 1, (
        f'wandb.finish must run via the finally block even on exception, '
        f'got call_count={wandb.finish.call_count}'
    )

    # Scenario 3 — raise_at=0 (very first step). Finish must still run.
    wandb.init.reset_mock(); wandb.finish.reset_mock()
    try:
        ex2_safe_train(n_steps=10, raise_at=0)
    except RuntimeError:
        pass
    assert wandb.finish.call_count == 1, 'finish must run even if loop raises on step 0'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex2_safe_train(n_steps: int, raise_at) -> int:
    wandb.init(project='arena', name='safe-run')
    completed = 0
    try:
        for step in range(n_steps):
            if raise_at is not None and step == raise_at:
                raise RuntimeError(f'simulated failure at step {step}')
            completed += 1
    finally:
        wandb.finish()
    return completed
```

**`finally` runs unconditionally.** Normal return, raised exception, even `return` inside the `try` — `finally` still runs. It is the right tool when you want "this cleanup MUST happen, no matter what."

**Don't catch + re-raise.** `try: ... except: wandb.finish(); raise` works but is more code and is wrong if you raise something the except clause doesn't catch. `finally` covers every exception class for free.

**Why not a context manager?** Wandb DOES expose a context-manager API (`with wandb.init(...) as run:`), which is nicer. ARENA's code base predates it, so the explicit try/finally is still the prevailing idiom in the codebase.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()